# Correctness checks: does our code do what Antoine's method says?

Before comparing any variation, the code is checked against the method, step by step. Each check states what
the method requires, measures it on the code's own output, and passes or fails (`checks.py`).

| # | What the method requires | How it is checked |
|---|---|---|
| 1 | Alarm when the window's outlier rate differs from the reference's by more than z(1-α/2)/√(2n) (ADAZOR) | the threshold for n = 250, α = 0.025 is 0.1002; our detector agrees with adazor's `ZTestDriftDetector` on 2,000 random windows |
| 2 | Starting graph = every triple up to the cutoff plus the undated ones, all types; stream = one type, in time order; reference = monitored triples before the last 250 | counts read with the run code's own loader, on all three data files |
| 3 | Window of 250; after an alarm keep 25, refill 225 untested, re-embed, new reference = window; re-embed at every triple with an unseen entity or relation; the graph grows by the stream; Rules 1, 2, 4 | runs on the real CASE1 stream with a random stand-in embedding (no training, so fast), every step compared with what it should be, recomputed independently from the data |
| 4 | Sphere rule: ratio = distance to the box centre / half the box diagonal | a point inside the box has ratio ≤ 1, a corner exactly 1 |
| 5 | Features: Hadamard s·p·o, concatenation [s, p, o], unit vectors | on known vectors |
| 6 | Warm start: old vectors kept, new entity random, a few epochs over the whole graph | real PyKEEN on a tiny graph, TransE and ComplEx: exact copy, then cosine of old vectors after one warm epoch vs from scratch |
| 7 | Our port reproduces Antoine's own run | his settings (`--preset antoine`: 32 dims, 2 epochs, lr 0.001, γ 0.5, no scaling) on CASE1 with TransE give his alarm file rows 2295, 2857, 3430, 37769 |

How each part of our code maps to Antoine's notebook (`reference/antoine_sphere_rule_edition2.ipynb`):

| Antoine's notebook | Our code (`drift_sphere.py`) | Difference |
|---|---|---|
| cell 1: `ZTestDriftDetector` (adazor) with fallback | `make_detector`, `FallbackZTestDriftDetector` | none |
| cell 2: `load_temporal_triples`, `sort_temporal_triples`, `build_initial_knowledge_graph`, `triple_has_unseen_elements`, `update_knowledge_graph` | same names | the graph is appended to instead of re-concatenated (same rows, same order) |
| cell 3: `generate_transe_embeddings_from_kg` | `generate_embeddings_from_kg` | 5 models instead of TransE; learning rate and epochs from the version; warm start (v3) |
| cell 4: `create_hadamard_features`, `create_concatenation_features`, `train_classifier`, `retrain_reference_model`, box and ratio functions | `compose`, `train_classifier`, `retrain_reference_model`, `boundary_from_min_max`, `compute_center_radius_metrics` | vectorised (same numbers); ratio limit k is a setting (Antoine: 1) |
| cell 5: `process_triple_for_features`, `rebuild_reference_after_drift`, `run_dynamic_ocsvm_ztest_stream_keep10` | same names, `run_stream` | Rules 2, 3, 4 added; logging of the untested refill triples (logging only) |

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import pandas as pd
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60); pd.set_option('display.max_rows', 400)
HERE = Path.cwd() if (Path.cwd() / 'results_helpers.py').exists() else Path.cwd() / 'notebooks'
sys.path.insert(0, str(HERE))
import results_helpers as rh

import subprocess
RUN_CHECKS = True   # False: only read checks/checks.csv from the last run
if RUN_CHECKS:
    out = subprocess.run([sys.executable, str(rh.PHASE1 / 'checks.py')], cwd=rh.PHASE1, capture_output=True, text=True)
    print(out.stdout[-3000:] if out.returncode == 0 else out.stdout[-3000:] + out.stderr[-3000:])

In [ ]:
c = rh.read_csv(rh.paths()['checks'])
print(f"{len(c)} checks: {int(c['passed'].sum())} passed, {int((~c['passed']).sum())} failed")
c[['check', 'what', 'expected', 'got', 'passed']]

In [ ]:
c[~c['passed']]   # anything listed here needs attention before the results are read